In [1]:
import os
import cv2


def read_images_labels(folder_path):
    images = []
    labels = []
    label_path = os.path.join(folder_path, "labels")
    image_path = os.path.join(folder_path, "images")
    label_files = os.listdir(label_path)

    for label_fname in label_files:
        label_file_path = os.path.join(label_path, label_fname)
        if not os.path.isfile(label_file_path):
            continue
        with open(label_file_path, "r") as f:
            data = f.read()
        label = int(data.split()[0])
        image_file_path = os.path.join(image_path, label_fname.replace(".txt", ".jpg"))
        image = cv2.imread(image_file_path)
        if image is not None:
            images.append(image)
            labels.append(label)

    return images, labels


In [5]:
%cd Color.v1i.yolov9/


c:\Users\jason\Desktop\cam_server\FYP\KNN_train\Color.v1i.yolov9


c:\Users\jason\Desktop\cam_server\FYP\KNN_train\env\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
train_np = read_images_labels("./")


In [7]:
print(len(train_np[1]))


6244


In [8]:
def image_to_histogram(image, bins=32):
    histogram = []
    for i in range(3):  # Assuming the image is in BGR format
        hist = cv2.calcHist([image], [i], None, [bins], [0, 256])
        histogram.extend(hist.flatten())
    return histogram


In [9]:
def prepare_dataset(images):
    data = [image_to_histogram(image) for image in images]
    return data


In [10]:
data = prepare_dataset(train_np[0])
labels = train_np[1]


In [11]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


def train_knn(data, labels):
    X_train, X_test, y_train, y_test = train_test_split(
        data, labels, test_size=0.2, random_state=42
    )

    # Encode labels
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    y_test_encoded = le.transform(y_test)

    # Initialize and train KNN
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train, y_train_encoded)

    # Test accuracy
    accuracy = knn.score(X_test, y_test_encoded)
    print(f"Test accuracy: {accuracy}")

    return knn, le


In [12]:
model = train_knn(data, labels)


Test accuracy: 0.9383506805444356


In [13]:
import joblib

joblib.dump(model, "helmet_color_cls.pkl")


['helmet_color_cls.pkl']

In [14]:
model[0].predict([data[0]])


array([0], dtype=int64)

In [15]:
def predict_image(image, knn_model, label_encoder, bins=32):
    histogram = image_to_histogram(image, bins)
    prediction = knn_model.predict([histogram])
    predicted_label = label_encoder.inverse_transform(prediction)
    return predicted_label


In [19]:
import joblib

img = cv2.imread("../white.png")



load_model = joblib.load("helmet_color_cls.pkl", mmap_mode="r")



# result = predict_image(train_np[0][6565], model[0], model[1])



result = predict_image(img, load_model[0], load_model[1])



result


array([2])

# Save model


In [ ]:
import joblib

joblib.dump(model, "helmet_color_cls.pkl")


In [7]:
%cd Color.v1i.yolov9/


c:\Users\jason\Desktop\cam_server\FYP\KNN_train\Color.v1i.yolov9


c:\Users\jason\Desktop\cam_server\FYP\KNN_train\env\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [9]:
# %cd Color.v1i.yolov9



c:\Users\jason\Desktop\cam_server\FYP\KNN_train\Color.v1i.yolov9


c:\Users\jason\Desktop\cam_server\FYP\KNN_train\env\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [12]:
knn = joblib.load("helmet_color_cls.pkl", mmap_mode="r")
